# CloudCommerce Policy RAG

End-to-end notebook for testing the policy retrieval and answer pipeline.

**Flow**

1. Load Markdown policies with `TextLoader`
2. Split by Markdown headers and chunk size
3. Index chunks in persistent Chroma
4. Retrieve relevant policy sections
5. Answer through the official retrieval chain
6. Fall back to `I don't know` when no policy matches

**Requirements**

- The first code cell locates the project root automatically
- `.env` must contain your provider and API key settings
- Restart the kernel after changing `.env`

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not find the project root (pyproject.toml).")
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
from collections import Counter

from app.config import get_settings
from app.rag import (
    UNKNOWN_ANSWER,
    answer_policy_question,
    build_policy_chain,
    build_policy_retriever,
    build_policy_store,
    citation_for,
    format_citations,
    load_policy_documents,
)

## 1. Configuration

In [ ]:
settings = get_settings()

print("Policy docs:", settings.policy_docs_path)
print("Chroma path:", settings.chroma_persist_path)
print("Collection:", settings.chroma_collection_name)
print("Chunk size:", settings.rag_chunk_size)
print("Chunk overlap:", settings.rag_chunk_overlap)
print("Top-k:", settings.rag_top_k)
print("Min score:", settings.rag_min_score)

## 2. Load and split policies

`TextLoader` reads each Markdown file. `MarkdownHeaderTextSplitter` keeps the
`title` and `section` metadata, and `RecursiveCharacterTextSplitter` controls
chunk size.

In [ ]:
documents = load_policy_documents(
    settings.policy_docs_path,
    chunk_size=settings.rag_chunk_size,
    chunk_overlap=settings.rag_chunk_overlap,
)

print(f"Loaded {len(documents)} chunks")
print(Counter(document.metadata["source"] for document in documents))
print()
print(documents[0].metadata)
print()
print(documents[0].page_content[:300])

## 3. Build the Chroma index

In [ ]:
store = build_policy_store(settings)
print(f"Indexed chunks: {len(store.get()['ids'])}")

## 4. Create the retriever

In [ ]:
retriever = build_policy_retriever(settings)
retriever

## 5. Retrieve policy chunks

In [ ]:
questions = [
    "Can I return an item after 14 days?",
    "Can I cancel an order that is still processing?",
    "How long do refunds take?",
]

for question in questions:
    matches = retriever.invoke(question)
    print(f"\nQ: {question}  ({len(matches)} matches)")
    for document in matches:
        print(f"- {citation_for(document)}")

### Relevance scores

In [ ]:
for question in questions:
    results = store.similarity_search_with_relevance_scores(
        question,
        k=settings.rag_top_k,
    )
    print(f"\nQ: {question}")
    for document, score in results:
        print(f"  {score:.3f}  {citation_for(document)}")

## 6. Build the RAG chain

In [ ]:
chain = build_policy_chain(settings)
response = chain.invoke({"input": "How long do refunds take?"})

print(response["answer"])
print()
print(format_citations(response["context"]))

## 7. Ask a question with citations

In [ ]:
def ask(question: str) -> None:
    response = answer_policy_question(question)
    print(f"Q: {question}")
    print(f"A: {response['answer']}")

    citations = format_citations(response.get("context", []))
    if citations:
        print(citations)


ask("Can I cancel an order that is still processing?")

## 8. Uncovered question

No policy chunks pass the score threshold, so the pipeline returns a
deterministic fallback instead of calling the model.

In [ ]:
response = answer_policy_question("Do you offer student discounts?")
print(response["answer"])
print("Matches:", len(response["context"]))
assert response["answer"] == UNKNOWN_ANSWER

## 9. Rebuild the index (optional)

Uncomment and run this cell after changing policy files or chunk settings.
Restart the kernel afterwards so cached settings and retrievers are refreshed.

In [ ]:
# import shutil
# from app.rag import get_policy_chain, get_policy_retriever
#
# shutil.rmtree(settings.chroma_persist_path, ignore_errors=True)
# get_policy_retriever.cache_clear()
# get_policy_chain.cache_clear()
#
# print("Index removed. Re-run the index and chain cells.")